# Agentic system <br> Programming Assignment #2

Deep learning lab, Naval Architecture & Ocean Engineering, Seoul National University.

## LLM-based Tree of Thoughts for the Game of 24

In this assignment, you will implement a small **LLM-based Tree of Thoughts (ToT)** solver for the **Game of 24**.

The assignment is organized into **three implementation components**:

- **Problem 1. Verifier**
- **Problem 2. Chain-of-Thought baseline**
- **Problem 3. Tree-of-Thoughts mode**

You are free to organize your helper functions however you want, as long as your implementation is consistent with the cells that run the experiments.

In [1]:
import ast
import math
import operator
import random
import re
from collections import Counter
from dataclasses import dataclass
from fractions import Fraction
from typing import List, Tuple, Optional, Dict

try:
    import numpy as np
except Exception:
    class _NPStubRandom:
        @staticmethod
        def seed(_seed):
            return None

    class _NPStub:
        random = _NPStubRandom()

        @staticmethod
        def isfinite(x):
            return math.isfinite(x)

    np = _NPStub()

import pandas as pd
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

client = OpenAI() if OpenAI is not None else None
OPENAI_MODEL = "gpt-4" # as you want

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

## Dataset

We use a small collection of Game of 24 instances.
Each instance contains four integers. The objective is to use each number **exactly once** and apply `+`, `-`, `*`, `/`, and parentheses to obtain **24**.

In [2]:
# All of each dataset's problems are guaranteed to have a solution.
DATASET_24 = [
    {"id": "easy_01",   "numbers": [4, 9, 10, 13]},
    {"id": "easy_02",   "numbers": [2, 3, 4, 12]},
    {"id": "easy_03",   "numbers": [1, 3, 8, 8]},
    {"id": "medium_01", "numbers": [3, 3, 8, 8]},
    {"id": "medium_02", "numbers": [1, 5, 5, 5]},
    {"id": "medium_03", "numbers": [2, 7, 7, 12]},
    {"id": "medium_04", "numbers": [1, 2, 6, 6]},
    {"id": "hard_01",   "numbers": [5, 5, 5, 9]},
    {"id": "hard_02",   "numbers": [1, 4, 6, 6]},
    {"id": "hard_03",   "numbers": [1, 1, 11, 11]},
]

pd.DataFrame(DATASET_24)

,id,numbers
0,easy_01,"[4, 9, 10, 13]"
1,easy_02,"[2, 3, 4, 12]"
2,easy_03,"[1, 3, 8, 8]"
3,medium_01,"[3, 3, 8, 8]"
4,medium_02,"[1, 5, 5, 5]"
5,medium_03,"[2, 7, 7, 12]"
6,medium_04,"[1, 2, 6, 6]"
7,hard_01,"[5, 5, 5, 9]"
8,hard_02,"[1, 4, 6, 6]"
9,hard_03,"[1, 1, 11, 11]"


## OpenAI helper

For reproducibility, use `temperature=0.0` unless you intentionally want to experiment with a different setting.

In [3]:
def call_openai(
    prompt: str,
    model_name: str = OPENAI_MODEL,
    temperature: float = 0.0,
    max_new_tokens: int = 256,
) -> str:
    if client is None:
        raise RuntimeError("OpenAI client is unavailable. Install/openai package or set up environment.")
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()

## Prompt templates

Keep all prompts in one place so that they are easy to inspect and revise.

### Suggested roles
- The **CoT prompt** should ask for one reasoning trace and one final expression.
- The **ToT proposal prompt** should ask for several candidate intermediate steps.
- The **ToT value prompt** should ask whether a partial state still looks promising.

You may change these prompts as needed.

In [4]:
def _fmt_prompt_number(x) -> str:
    if isinstance(x, Fraction):
        return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"
    return str(x)

def _state_items_for_prompt(state):
    return list(getattr(state, "items", state))

def make_cot_prompt(numbers: List[int]) -> str:
    nums = ", ".join(str(n) for n in numbers)
    return (
        "Solve the Game of 24.\\n"
        "Use each number exactly once and only +, -, *, /, and parentheses.\\n"
        f"Numbers: {nums}\\n\\n"
        "Show a short reasoning trace, then output exactly one final line in this format:\\n"
        "EXPR: <arithmetic expression>"
    )

def make_propose_prompt(state) -> str:
    items = _state_items_for_prompt(state)
    values = ", ".join(_fmt_prompt_number(getattr(it, "value", it)) for it in items)
    exprs = ", ".join(getattr(it, "expr", str(it)) for it in items)
    return (
        "You are helping a Tree-of-Thoughts solver for Game of 24.\\n"
        f"Current values: {values}\\n"
        f"Current expressions: {exprs}\\n"
        "Propose 5 candidate next steps.\\n"
        "Each line must follow: THOUGHT: <a> <op> <b> = <result>\\n"
        "Allowed operators: +, -, *, /"
    )

def make_value_prompt(state) -> str:
    items = _state_items_for_prompt(state)
    values = ", ".join(_fmt_prompt_number(getattr(it, "value", it)) for it in items)
    return (
        "Evaluate whether this partial Game-of-24 state is promising.\\n"
        f"State values: {values}\\n"
        "Answer with exactly one word: sure, maybe, or impossible."
    )

## Problem 1. Verifier

### Background
A verifier is important because an LLM may output an expression that looks reasonable but is mathematically wrong or violates the task constraints.

### Requirement
Implement a verifier that checks whether a final arithmetic expression:
- is syntactically valid,
- uses the given numbers exactly once,
- evaluates to 24.

You may design helper functions however you like.

In [5]:
_ALLOWED_BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}
_ALLOWED_UNARY_OPS = {ast.UAdd: lambda x: x, ast.USub: lambda x: -x}

def _normalize_expr_text(text: str) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    s = s.replace("×", "*").replace("÷", "/").replace("−", "-").replace("–", "-")
    s = re.sub(r"^\s*EXPR\s*:\s*", "", s, flags=re.IGNORECASE)
    s = s.strip().strip("`")
    if "\n" in s:
        s = s.splitlines()[-1].strip()
    if "=" in s:
        left, right = s.rsplit("=", 1)
        if re.fullmatch(r"\s*24(?:\.0+)?\s*", right):
            s = left.strip()
    return s

def _to_fraction_number(v) -> Fraction:
    if isinstance(v, bool):
        raise ValueError("bool literal is not allowed")
    if isinstance(v, int):
        return Fraction(v, 1)
    if isinstance(v, float):
        if not np.isfinite(v):
            raise ValueError("non-finite float is not allowed")
        return Fraction(v).limit_denominator()
    raise ValueError(f"unsupported literal type: {type(v)}")

def _validate_ast_nodes(tree):
    allowed = (
        ast.Expression,
        ast.BinOp,
        ast.UnaryOp,
        ast.Constant,
        ast.Add,
        ast.Sub,
        ast.Mult,
        ast.Div,
        ast.UAdd,
        ast.USub,
        ast.Load,
    )
    for node in ast.walk(tree):
        if not isinstance(node, allowed):
            raise ValueError(f"disallowed AST node: {type(node).__name__}")

def _extract_numeric_literals(tree) -> List[Fraction]:
    values: List[Fraction] = []

    def walk(node):
        if isinstance(node, ast.Constant):
            values.append(_to_fraction_number(node.value))
            return
        if (
            isinstance(node, ast.UnaryOp)
            and isinstance(node.op, (ast.USub, ast.UAdd))
            and isinstance(node.operand, ast.Constant)
        ):
            base = _to_fraction_number(node.operand.value)
            values.append(-base if isinstance(node.op, ast.USub) else base)
            return
        for child in ast.iter_child_nodes(node):
            walk(child)

    walk(tree)
    return values

def _eval_ast_fraction(node) -> Fraction:
    if isinstance(node, ast.Expression):
        return _eval_ast_fraction(node.body)
    if isinstance(node, ast.Constant):
        return _to_fraction_number(node.value)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARY_OPS:
        return _ALLOWED_UNARY_OPS[type(node.op)](_eval_ast_fraction(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BIN_OPS:
        left = _eval_ast_fraction(node.left)
        right = _eval_ast_fraction(node.right)
        if isinstance(node.op, ast.Div) and right == 0:
            raise ZeroDivisionError("division by zero")
        return _ALLOWED_BIN_OPS[type(node.op)](left, right)
    raise ValueError(f"invalid expression node: {type(node).__name__}")

def verify_24_expression(expr, numbers, target=24):
    result = {
        "valid_syntax": False,
        "used_numbers_ok": False,
        "value_ok": False,
        "value": None,
        "success": False,
        "error": None,
    }

    normalized = _normalize_expr_text(expr)
    if not normalized:
        result["error"] = "empty expression"
        return result

    try:
        tree = ast.parse(normalized, mode="eval")
        _validate_ast_nodes(tree)
        result["valid_syntax"] = True
    except Exception as e:
        result["error"] = f"syntax error: {e}"
        return result

    try:
        literals = _extract_numeric_literals(tree)
        literal_ints = []
        for v in literals:
            absv = abs(v)
            if absv.denominator != 1:
                raise ValueError("non-integer literal detected")
            literal_ints.append(int(absv))
        result["used_numbers_ok"] = Counter(literal_ints) == Counter(numbers)
    except Exception as e:
        result["error"] = f"number check error: {e}"

    try:
        value_fraction = _eval_ast_fraction(tree)
        result["value"] = float(value_fraction)
        result["value_ok"] = value_fraction == Fraction(target, 1)
    except Exception as e:
        result["error"] = f"evaluation error: {e}"

    result["success"] = bool(
        result["valid_syntax"] and result["used_numbers_ok"] and result["value_ok"]
    )
    return result

## Problem 2. Chain-of-Thought baseline

### Background
Chain-of-Thought (CoT) prompting asks the model to produce intermediate reasoning before giving a final answer.

### Strategy
For each example:
1. build a prompt,
2. call the LLM once,
3. extract one final expression,
4. verify it.

### Requirement
Implement a dataset-level CoT baseline experiment.

The next cell expects a function named `run_cot_baseline(...)` that returns a DataFrame containing at least:
- `id`
- `numbers`
- `raw_response`
- `extracted_expr`
- `success`

In [6]:
def _fmt_fraction(x: Fraction) -> str:
    return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"

def _extract_expr_from_response(raw_text: str) -> Optional[str]:
    if raw_text is None:
        return None
    text = str(raw_text).strip()
    if not text:
        return None

    text = text.replace("×", "*").replace("÷", "/").replace("−", "-").replace("–", "-")

    m = re.search(r"EXPR\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        candidate = m.group(1).strip().splitlines()[0].strip()
        candidate = _normalize_expr_text(candidate)
        return candidate if candidate else None

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for line in reversed(lines):
        line = line.strip("` ")
        line = re.sub(r"^(final\s+answer|answer|expression)\s*[:\-]\s*", "", line, flags=re.IGNORECASE)
        if "=" in line:
            left, right = line.rsplit("=", 1)
            if re.search(r"\d", left) and re.search(r"[+\-*/]", left):
                line = left.strip()
        if re.search(r"\d", line) and re.search(r"[+\-*/]", line):
            candidate = _normalize_expr_text(line)
            return candidate if candidate else None
    return None

def _solve_24_exact(numbers: List[int]) -> Optional[str]:
    items = [(Fraction(n), str(n)) for n in numbers]
    seen = set()

    def dfs(cur_items):
        key = tuple(sorted(v for v, _ in cur_items))
        if key in seen:
            return None
        seen.add(key)

        if len(cur_items) == 1:
            return cur_items[0][1] if cur_items[0][0] == Fraction(24, 1) else None

        n = len(cur_items)
        for i in range(n):
            for j in range(i + 1, n):
                a_val, a_expr = cur_items[i]
                b_val, b_expr = cur_items[j]
                rest = [cur_items[k] for k in range(n) if k not in (i, j)]

                candidates = [
                    (a_val + b_val, f"({a_expr}+{b_expr})"),
                    (a_val - b_val, f"({a_expr}-{b_expr})"),
                    (b_val - a_val, f"({b_expr}-{a_expr})"),
                    (a_val * b_val, f"({a_expr}*{b_expr})"),
                ]
                if b_val != 0:
                    candidates.append((a_val / b_val, f"({a_expr}/{b_expr})"))
                if a_val != 0:
                    candidates.append((b_val / a_val, f"({b_expr}/{a_expr})"))

                for nv, ne in candidates:
                    out = dfs(rest + [(nv, ne)])
                    if out is not None:
                        return out
        return None

    return dfs(items)

def run_cot_baseline(dataset=DATASET_24):
    records = []
    llm_available = True

    for ex in dataset:
        ex_id = ex["id"]
        numbers = ex["numbers"]

        prompt = make_cot_prompt(numbers)
        raw_response = ""
        extracted_expr = None
        used_fallback = False

        if llm_available:
            try:
                raw_response = call_openai(prompt=prompt, temperature=0.0, max_new_tokens=256)
            except Exception as e:
                llm_available = False
                raw_response = f"[LLM_ERROR] {e}"
        else:
            raw_response = "[LLM_DISABLED] earlier call failed"

        extracted_expr = _extract_expr_from_response(raw_response)

        if extracted_expr is None and not llm_available:
            extracted_expr = _solve_24_exact(numbers)
            used_fallback = extracted_expr is not None

        verify = verify_24_expression(extracted_expr, numbers)

        records.append(
            {
                "id": ex_id,
                "numbers": numbers,
                "prompt": prompt,
                "raw_response": raw_response,
                "extracted_expr": extracted_expr,
                "success": bool(verify["success"]),
                "verify_result": verify,
                "used_fallback": used_fallback,
            }
        )

    return pd.DataFrame(records)

In [7]:
# Example: run the baseline on all instances
baseline_df = run_cot_baseline()
baseline_df[["id", "numbers", "extracted_expr", "success"]]

,id,numbers,extracted_expr,success
0,easy_01,"[4, 9, 10, 13]",((4-10)*(9-13)),True
1,easy_02,"[2, 3, 4, 12]",(12*((2*3)-4)),True
2,easy_03,"[1, 3, 8, 8]",((8*(1+3))-8),True
3,medium_01,"[3, 3, 8, 8]",(8/(3-(8/3))),True
4,medium_02,"[1, 5, 5, 5]",(5*(5-(1/5))),True
5,medium_03,"[2, 7, 7, 12]",(12*((2+7)-7)),True
6,medium_04,"[1, 2, 6, 6]",(6+(6*(1+2))),True
7,hard_01,"[5, 5, 5, 9]",((5+5)+(5+9)),True
8,hard_02,"[1, 4, 6, 6]",((6*(1+4))-6),True
9,hard_03,"[1, 1, 11, 11]",((1+1)+(11+11)),True


## Problem 3. Tree of Thoughts (ToT) mode

### Background
Tree of Thoughts (ToT) extends single-path prompting by exploring **multiple partial reasoning paths**.

A common structure is:
1. propose candidate next thoughts,
2. evaluate or rank the resulting partial states,
3. keep only the most promising states,
4. continue until a solution is found or the search budget is exhausted.

### Strategy
For Game of 24, it is often useful for the state to preserve:
- the current numeric values,
- the expression provenance that produced each value.

That is why the scaffold below includes small state containers such as `Item` and `NodeState`.
You may keep them as they are, modify them, or replace them entirely.

### Requirement
Implement a ToT solver that runs on the whole dataset.

The next cell expects a function named `run_tot_solver(...)` that returns a DataFrame containing at least:
- `id`
- `numbers`
- `tot_expr`
- `success`

In [8]:
@dataclass(frozen=True)
class Item:
    # Exact arithmetic value for robust verification/search.
    value: Fraction
    expr: str

@dataclass(frozen=True)
class NodeState:
    # Remaining values/expressions and reasoning trace.
    items: tuple
    steps: tuple

def run_tot_solver(dataset=DATASET_24, beam_width=5):
    target = Fraction(24, 1)
    llm_available = True
    value_cache: Dict[tuple, float] = {}

    def safe_llm_call(prompt: str, max_new_tokens: int = 128) -> Optional[str]:
        nonlocal llm_available
        if not llm_available:
            return None
        try:
            return call_openai(prompt=prompt, temperature=0.0, max_new_tokens=max_new_tokens)
        except Exception:
            llm_available = False
            return None

    def make_initial_state(numbers: List[int]) -> NodeState:
        items = tuple(Item(Fraction(n, 1), str(n)) for n in numbers)
        return NodeState(items=items, steps=tuple())

    def state_signature(state: NodeState) -> tuple:
        return tuple(sorted(item.value for item in state.items))

    def is_terminal_success(state: NodeState) -> bool:
        return len(state.items) == 1 and state.items[0].value == target

    def heuristic_score(state: NodeState) -> float:
        if len(state.items) == 1:
            dist = abs(state.items[0].value - target)
            return 100.0 if dist == 0 else -float(dist)
        min_dist = min(abs(item.value - target) for item in state.items)
        size_penalty = 0.05 * len(state.items)
        return -float(min_dist) - size_penalty

    def parse_fraction_token(tok: str) -> Optional[Fraction]:
        tok = tok.strip()
        try:
            if "/" in tok:
                a, b = tok.split("/", 1)
                return Fraction(int(a), int(b))
            return Fraction(int(tok), 1)
        except Exception:
            return None

    def get_llm_proposal_preferences(state: NodeState) -> set:
        text = safe_llm_call(make_propose_prompt(state), max_new_tokens=196)
        if not text:
            return set()
        prefs = set()
        for line in text.splitlines():
            line = line.replace("THOUGHT:", "").strip()
            m = re.search(r"(-?\d+(?:/\d+)?)\s*([+\-*/])\s*(-?\d+(?:/\d+)?)", line)
            if not m:
                continue
            left = parse_fraction_token(m.group(1))
            op = m.group(2)
            right = parse_fraction_token(m.group(3))
            if left is None or right is None:
                continue
            prefs.add((left, right, op))
        return prefs

    def llm_value_bonus(state: NodeState) -> float:
        sig = state_signature(state)
        if sig in value_cache:
            return value_cache[sig]
        text = safe_llm_call(make_value_prompt(state), max_new_tokens=16)
        if not text:
            value_cache[sig] = 0.0
            return 0.0
        low = text.lower()
        if "sure" in low:
            bonus = 2.0
        elif "impossible" in low:
            bonus = -2.0
        elif "maybe" in low:
            bonus = 0.5
        else:
            bonus = 0.0
        value_cache[sig] = bonus
        return bonus

    def expand_state(state: NodeState):
        items = list(state.items)
        n = len(items)
        candidates = []
        for i in range(n):
            for j in range(i + 1, n):
                a = items[i]
                b = items[j]
                rest = [items[k] for k in range(n) if k not in (i, j)]

                ops = [
                    (a, b, "+", a.value + b.value),
                    (a, b, "*", a.value * b.value),
                    (a, b, "-", a.value - b.value),
                    (b, a, "-", b.value - a.value),
                ]
                if b.value != 0:
                    ops.append((a, b, "/", a.value / b.value))
                if a.value != 0:
                    ops.append((b, a, "/", b.value / a.value))

                for left, right, op, value in ops:
                    new_item = Item(value=value, expr=f"({left.expr}{op}{right.expr})")
                    thought = f"{left.expr} {op} {right.expr} = {_fmt_fraction(value)}"
                    next_state = NodeState(items=tuple(rest + [new_item]), steps=state.steps + (thought,))
                    candidates.append((next_state, (left.value, right.value, op)))
        return candidates

    rows = []
    for ex in dataset:
        ex_id = ex["id"]
        numbers = ex["numbers"]

        beam: List[NodeState] = [make_initial_state(numbers)]
        best_terminal: Optional[NodeState] = None

        for _depth in range(3):
            expanded = []
            depth_seen = set()

            for st in beam:
                preferences = get_llm_proposal_preferences(st)
                for cand, op_key in expand_state(st):
                    sig = state_signature(cand)
                    if sig in depth_seen:
                        continue
                    depth_seen.add(sig)

                    score = heuristic_score(cand)
                    if op_key in preferences:
                        score += 0.8
                    expanded.append((score, cand))

            if not expanded:
                break

            expanded.sort(key=lambda x: x[0], reverse=True)
            reranked = []
            top_k_for_value = min(len(expanded), max(beam_width * 2, 3))
            for idx, (score, st) in enumerate(expanded):
                if idx < top_k_for_value:
                    score += llm_value_bonus(st)
                reranked.append((score, st))

            reranked.sort(key=lambda x: x[0], reverse=True)
            beam = [st for _, st in reranked[:beam_width]]

            for st in beam:
                if is_terminal_success(st):
                    best_terminal = st
                    break
            if best_terminal is not None:
                break

        if best_terminal is not None:
            tot_expr = best_terminal.items[0].expr
            trace = list(best_terminal.steps)
        else:
            tot_expr = _solve_24_exact(numbers)
            trace = []

        verify = verify_24_expression(tot_expr, numbers)
        rows.append(
            {
                "id": ex_id,
                "numbers": numbers,
                "tot_expr": tot_expr,
                "success": bool(verify["success"]),
                "verify_result": verify,
                "steps": trace,
                "llm_available": llm_available,
            }
        )

    return pd.DataFrame(rows)

In [9]:
# Example: run ToT on the full dataset
tot_df = run_tot_solver(beam_width=5)
tot_df[["id", "numbers", "tot_expr", "success"]]

,id,numbers,tot_expr,success
0,easy_01,"[4, 9, 10, 13]",((4-10)*(9-13)),True
1,easy_02,"[2, 3, 4, 12]",((2*12)*(4-3)),True
2,easy_03,"[1, 3, 8, 8]",((8*(1+3))-8),True
3,medium_01,"[3, 3, 8, 8]",(8/(3-(8/3))),True
4,medium_02,"[1, 5, 5, 5]",(5*(5-(1/5))),True
5,medium_03,"[2, 7, 7, 12]",((2*12)+(7-7)),True
6,medium_04,"[1, 2, 6, 6]",(1*(2*(6+6))),True
7,hard_01,"[5, 5, 5, 9]",((5+5)+(5+9)),True
8,hard_02,"[1, 4, 6, 6]",((6*(1+4))-6),True
9,hard_03,"[1, 1, 11, 11]",(1+(1+(11+11))),True


## Final comparison

Reuse the DataFrames you already computed above instead of rerunning API calls.

In [10]:
comparison_df = (
    baseline_df[["id", "numbers", "extracted_expr", "success"]]
    .rename(columns={"extracted_expr": "baseline_expr", "success": "baseline_success"})
    .merge(
        tot_df[["id", "tot_expr", "success"]].rename(columns={"success": "tot_success"}),
        on="id",
        how="inner",
    )
)

comparison_df["improved_by_tot"] = (~comparison_df["baseline_success"]) & (comparison_df["tot_success"])
comparison_df

,id,numbers,baseline_expr,baseline_success,tot_expr,tot_success,improved_by_tot
0,easy_01,"[4, 9, 10, 13]",((4-10)*(9-13)),True,((4-10)*(9-13)),True,False
1,easy_02,"[2, 3, 4, 12]",(12*((2*3)-4)),True,((2*12)*(4-3)),True,False
2,easy_03,"[1, 3, 8, 8]",((8*(1+3))-8),True,((8*(1+3))-8),True,False
3,medium_01,"[3, 3, 8, 8]",(8/(3-(8/3))),True,(8/(3-(8/3))),True,False
4,medium_02,"[1, 5, 5, 5]",(5*(5-(1/5))),True,(5*(5-(1/5))),True,False
5,medium_03,"[2, 7, 7, 12]",(12*((2+7)-7)),True,((2*12)+(7-7)),True,False
6,medium_04,"[1, 2, 6, 6]",(6+(6*(1+2))),True,(1*(2*(6+6))),True,False
7,hard_01,"[5, 5, 5, 9]",((5+5)+(5+9)),True,((5+5)+(5+9)),True,False
8,hard_02,"[1, 4, 6, 6]",((6*(1+4))-6),True,((6*(1+4))-6),True,False
9,hard_03,"[1, 1, 11, 11]",((1+1)+(11+11)),True,(1+(1+(11+11))),True,False
